# Model Calibration & Validation — SWAN vs ERA5 Reanalysis

---

## Overview

Model validation is a critical step in any numerical modeling study. This notebook compares SWAN model output against ERA5 reanalysis wave data to assess the model's accuracy and identify potential calibration improvements.

The workflow has two phases:

### Phase 1 — Calibration (Parameter Sweep Framework)

Systematically vary three SWAN physics parameters and identify the combination that maximises Kling–Gupta Efficiency (KGE) against ERA5:

- $C_{ds}$ — whitecap dissipation coefficient (``GEN3 KOMen``)
- $C_f$   — bottom-friction coefficient (``FRIC JONswap``)
- $\gamma$ — depth-induced breaking parameter (``BREAK CONstant``)

In a full sweep, each of the $3^3 = 27$ combinations is run through SWAN and scored. This notebook implements the *framework* and, because re-running 27 SWAN jobs can take hours, **fills the non-baseline rows of the ranking table with a clearly labelled synthetic perturbation** so the ranking machinery can be demonstrated even when SWAN is not re-invoked. If you want the full sweep, set ``RUN_SWEEP = True`` in the sweep cell.

### Phase 2 — Validation (Time series, Scatter, Taylor, Metrics)

Compare the calibrated run against ERA5 using all four standard diagnostics and classify performance against literature thresholds.

This notebook is **region-agnostic** — set the environment variable ``SWAN_REGION`` (or the ``REGION`` variable in the first code cell) to ``california_bay`` or ``oman``.


> **What this comparison is (and is not).** ERA5 supplies this SWAN model's wave boundary conditions (the TPAR files) and its wind forcing, and it is also the reference data set used below. Agreement with ERA5 therefore shows that SWAN is **consistent with its forcing reanalysis**; it is *not* an independent validation. For independent validation compare against in-situ observations, e.g. NDBC buoys 46026 (San Francisco) and 46012 (Half Moon Bay) for California Bay, or satellite altimetry.


In [ ]:
import os
import sys
import shutil
import datetime
import itertools
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import netCDF4 as nc

%matplotlib inline

# ---------------------------------------------------------------------
# Region-aware configuration
# ---------------------------------------------------------------------
# --- bootstrap: locate the repository and make code/ importable ---
import os, sys
REPO_ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(REPO_ROOT, 'code')):
    if os.path.dirname(REPO_ROOT) == REPO_ROOT:
        raise FileNotFoundError('Repository root not found (no code/ directory). '
                                'Start Jupyter from inside the repository.')
    REPO_ROOT = os.path.dirname(REPO_ROOT)
CODE_DIR = os.path.join(REPO_ROOT, 'code')
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
# --- end bootstrap ---

from regions import get_region, summarize   # noqa: E402
import swan_runner                           # noqa: E402

REGION = os.environ.get('SWAN_REGION', 'california_bay')
cfg = get_region(REGION)
print(summarize(REGION))

# Per-region figures directory for this notebook
FIGS_DIR = cfg.figs_dir('06_model_validation')
print(f'Figures will be saved to: {FIGS_DIR}')


In [ ]:
# Paths derived from region config
MODEL_DIR  = cfg.model_dir
TABLE_FILE = os.path.join(MODEL_DIR, 'site_output.tab')
SITES_FILE = os.path.join(MODEL_DIR, 'candidate_sites.dat')
WAVE_NC    = cfg.wave_nc_path
SWN_TMPL   = os.path.join(MODEL_DIR, 'ahp_run.swn')

for label, fp in [('TABLE', TABLE_FILE), ('SITES', SITES_FILE),
                  ('ERA5 WAVE', WAVE_NC), ('SWAN template', SWN_TMPL)]:
    print(f"{label:15s}: {'OK' if os.path.exists(fp) else 'MISSING  <-- ' + fp}")


## 1. Validation Metrics

We define a function to compute six standard statistical metrics that quantify the agreement between simulated ($S$) and observed ($O$) wave heights.

The **Kling–Gupta Efficiency (KGE)** decomposes model performance into three components:
- $r$ — correlation (timing of peaks)
- $\beta = \bar{S}/\bar{O}$ — bias ratio (mean level)
- $\gamma = \sigma_S / \sigma_O$ — variability ratio (dynamic range)

$$\text{KGE} = 1 - \sqrt{(r-1)^2 + (\beta - 1)^2 + (\gamma - 1)^2}$$

A KGE > 0.5 is generally considered acceptable for hydrological/wave models.

In [3]:
def validation_metrics(sim, obs):
    """
    Compute six standard validation metrics.

    Parameters
    ----------
    sim, obs : 1-D arrays of equal length (NaN-free).

    Returns
    -------
    dict with keys: Bias, MAE, RMSE, CC, NSE, KGE
    """
    sim, obs = np.asarray(sim, float), np.asarray(obs, float)

    bias  = np.mean(sim - obs)
    mae   = np.mean(np.abs(sim - obs))
    rmse  = np.sqrt(np.mean((sim - obs) ** 2))
    cc    = np.corrcoef(sim, obs)[0, 1]
    nse   = 1 - np.sum((obs - sim) ** 2) / np.sum((obs - np.mean(obs)) ** 2)

    # KGE decomposition
    beta  = np.mean(sim) / np.mean(obs)
    gamma = np.std(sim)  / np.std(obs)
    kge   = 1 - np.sqrt((cc - 1)**2 + (beta - 1)**2 + (gamma - 1)**2)

    return dict(Bias=bias, MAE=mae, RMSE=rmse, CC=cc, NSE=nse, KGE=kge)

## 2. Data Loaders

### 2.1 ERA5 Wave Data

ERA5 is the ECMWF global reanalysis dataset. For wave validation, we use the **significant wave height** (`swh`) variable from the ERA5 wave product. The data is on a regular lat/lon grid with hourly temporal resolution.

The NetCDF file stores time as numeric offsets from a reference date. We convert these to Python `datetime` objects for alignment with SWAN output.

In [ ]:
def load_era5_wave(nc_path):
    """
    Load ERA5 hourly wave data. Works with both the legacy 'time' variable
    and the newer 'valid_time' variable used by the CDS API.

    Returns
    -------
    dict with {times, lat, lon, swh (t, lat, lon)}
    """
    if not os.path.exists(nc_path):
        raise FileNotFoundError(f"ERA5 wave NetCDF not found: {nc_path}")
    ds  = nc.Dataset(nc_path)
    lat_name  = next(k for k in ds.variables if k.lower() in ('latitude',  'lat'))
    lon_name  = next(k for k in ds.variables if k.lower() in ('longitude', 'lon'))
    time_name = next(k for k in ds.variables if k.lower() in ('time', 'valid_time'))

    lat = ds.variables[lat_name][:]
    lon = ds.variables[lon_name][:]
    swh = ds.variables['swh'][:]
    tvar = ds.variables[time_name]
    cal  = tvar.calendar if hasattr(tvar, 'calendar') else 'standard'
    times_cf = nc.num2date(tvar[:], tvar.units, cal)
    ds.close()

    times = [datetime.datetime(t.year, t.month, t.day,
                               t.hour, t.minute, t.second) for t in times_cf]
    return dict(times=times, lat=np.asarray(lat), lon=np.asarray(lon),
                swh=np.ma.filled(swh, np.nan))


### 2.2 SWAN TABLE Output

The SWAN TABLE output is parsed using the same logic as `swan_runner.read_table()`, but here we also reshape the data into `(n_times, n_sites)` and reconstruct the timestamp array.

In [ ]:
def load_swan_table(filepath, n_sites, t0=None, step_hours=None):
    """Parse SWAN TABLE output and reshape into (n_times, n_sites)."""
    raw_data = swan_runner.read_table(filepath)

    n_total = len(raw_data['Hsig'])
    if n_total % n_sites:
        raise ValueError(
            f"TABLE row count ({n_total}) is not divisible by the number of "
            f"sites ({n_sites}). The TABLE output is out of sync with "
            "candidate_sites.dat."
        )
    n_times = n_total // n_sites
    data = {v: raw_data[v].reshape(n_times, n_sites) for v in raw_data}

    t0 = t0 or cfg.t_start
    step_hours = step_hours if step_hours is not None else cfg.t_step_hours
    times = [t0 + datetime.timedelta(hours=step_hours * h) for h in range(n_times)]
    print(f"  SWAN TABLE: {n_times} timesteps x {n_sites} sites, "
          f"variables: {list(data.keys())}")
    return times, data


## 3. Spatial Matching: ERA5 <-> SWAN

ERA5 and SWAN operate on different grids. To compare them, we need to **spatially match** ERA5 grid points to nearby SWAN candidate sites.

For each ERA5 grid point within the SWAN domain:
1. Find all SWAN candidate sites within a **0.25° search radius** (~25 km)
2. Average valid (ocean) SWAN $H_s$ values across those nearby sites
3. Pair the ERA5 observed $H_s$ with the SWAN simulated average at each common timestep

This produces a set of **matched (obs, sim)** pairs at each ERA5 comparison point.

In [ ]:
def match_era5_swan(era5, swan_times, swan_hs, site_locs,
                    search_radius=0.25, min_pairs=10):
    """
    For each ERA5 grid point near the SWAN domain, average all valid
    SWAN candidate sites within +/-search_radius deg and pair
    time-matched Hs values.

    Handles ERA5's 0..360 longitude convention (SWAN typically uses -180..180).

    Returns
    -------
    matched : list of dicts, each with
        {era5_lon, era5_lat, site_idx, n_sites_averaged, obs, sim, times}
    """
    era5_lat = np.asarray(era5['lat'])
    era5_lon_raw = np.asarray(era5['lon'])
    # Convert 0..360 to -180..180 if the SWAN domain uses negative longitudes
    if site_locs[:, 0].min() < 0 and era5_lon_raw.max() > 180:
        era5_lon = np.where(era5_lon_raw > 180, era5_lon_raw - 360, era5_lon_raw)
    else:
        era5_lon = era5_lon_raw
    era5_swh = era5['swh']
    era5_t   = era5['times']

    # SWAN domain extent with margin
    margin = 0.3
    lon_min = site_locs[:, 0].min() - margin
    lon_max = site_locs[:, 0].max() + margin
    lat_min = site_locs[:, 1].min() - margin
    lat_max = site_locs[:, 1].max() + margin

    swan_tdict = {t: i for i, t in enumerate(swan_times)}

    matched = []
    for ilat, la in enumerate(era5_lat):
        for ilon, lo in enumerate(era5_lon):
            if not (lat_min <= la <= lat_max and lon_min <= lo <= lon_max):
                continue

            dists = np.sqrt((site_locs[:, 0] - lo) ** 2
                            + (site_locs[:, 1] - la) ** 2)
            nearby = np.where(dists <= search_radius)[0]
            if len(nearby) == 0:
                nearby = [int(np.argmin(dists))]

            obs_list, sim_list, t_list = [], [], []
            for it_era, t_era in enumerate(era5_t):
                if t_era not in swan_tdict:
                    continue
                it_sw = swan_tdict[t_era]
                obs_val = era5_swh[it_era, ilat, ilon]
                if np.isnan(obs_val) or obs_val <= 0:
                    continue

                site_vals = swan_hs[it_sw, nearby]
                valid = site_vals > 0
                if valid.sum() == 0:
                    continue
                sim_avg = float(np.mean(site_vals[valid]))

                obs_list.append(float(obs_val))
                sim_list.append(sim_avg)
                t_list.append(t_era)

            if len(obs_list) > min_pairs:
                matched.append(dict(
                    era5_lon=float(lo), era5_lat=float(la),
                    site_idx=int(nearby[0]),
                    n_sites_averaged=len(nearby),
                    obs=np.array(obs_list), sim=np.array(sim_list),
                    times=t_list))

    return matched


## 4. Load Data

Load the ERA5 observations, SWAN TABLE output, and candidate site locations.

In [ ]:
# Load site locations
if not os.path.exists(SITES_FILE):
    raise FileNotFoundError(
        f"candidate_sites.dat not found in {MODEL_DIR}. "
        "Run notebooks/03_site_selection/swan_coupled_ahp_topsis.ipynb first."
    )
site_locs = np.loadtxt(SITES_FILE)
n_sites = site_locs.shape[0]
print(f"Loaded {n_sites} candidate sites")

# Load ERA5 observations
print("\nLoading ERA5 wave observations...")
era5 = load_era5_wave(WAVE_NC)
print(f"  ERA5: {len(era5['times'])} timesteps, "
      f"lat {era5['lat'].min():.1f}–{era5['lat'].max():.1f}, "
      f"lon {era5['lon'].min():.1f}–{era5['lon'].max():.1f}")

# Load SWAN TABLE output
print("\nLoading SWAN TABLE output...")
swan_times, swan_data = load_swan_table(TABLE_FILE, n_sites)
swan_hs = swan_data['Hsig']


## 5. Phase 1: Calibration Parameter Sweep

SWAN exposes three particularly influential physics parameters that often need tuning for coastal wave problems:

- $C_{ds}$ — whitecap dissipation (``GEN3 KOMen``)
- $C_f$   — JONSWAP bottom friction (``FRIC JONswap``)
- $\gamma$ — depth-induced breaking (``BREAK CONstant 1. γ``)

A full sweep would run SWAN 27 times (one per combination), match against ERA5, and rank by KGE. Because that takes hours, the default mode below is **demonstration**: the baseline row uses the *real* overall metrics, and the other 26 rows are filled with a clearly labelled **synthetic perturbation** so the ranking machinery and table layout can still be shown. Set ``RUN_SWEEP = True`` if you want the full run — the framework will iterate by calling ``swan_runner.prepare_input`` and ``swan_runner.run_swan`` for each combination.

Setting **RUN_SWEEP = True** runs all 26 non-baseline combinations for real: each one writes calib_NN.swn (with Cds, Cf and gamma substituted and the TABLE redirected to calib_NN.tab), runs SWAN, and scores the output against ERA5. It needs the ahp_run.swn written by the site-selection notebook.


In [ ]:
CALIB_PARAMS = {
    'Cds':   [1.0e-5, 1.36e-5, 2.36e-5],
    'Cf':    [0.019,  0.038,   0.067],
    'gamma': [0.65,   0.73,    0.80],
}

param_combos = list(itertools.product(
    CALIB_PARAMS['Cds'], CALIB_PARAMS['Cf'], CALIB_PARAMS['gamma']))

# Default: do NOT re-run SWAN 27 times. Flip to True if you want the full sweep.
RUN_SWEEP = False

print(f"Calibration parameter space: {len(param_combos)} combinations")
print(f"  Cds:   {CALIB_PARAMS['Cds']}")
print(f"  Cf:    {CALIB_PARAMS['Cf']}")
print(f"  gamma: {CALIB_PARAMS['gamma']}")

swan_available = shutil.which('swanrun') is not None
if swan_available:
    print("\nSWAN detected on PATH.")
else:
    print("\nSWAN NOT found on PATH -- full sweep is unavailable.")
if not RUN_SWEEP:
    print("\nRUN_SWEEP=False -- using existing site_output.tab as the baseline,")
    print("non-baseline rows will be clearly labelled as SYNTHETIC perturbations.")

print("\nExample SWAN input modifications for 3 of 27 combos:")
for i, (cds, cf, gm) in enumerate(param_combos[:3]):
    print(f"  [{i+1}]  GEN3 KOMen {cds:.2e}  |  "
          f"FRIC JONswap {cf:.3f}  |  "
          f"BREAK CONstant 1. {gm:.2f}")


### Spatial Matching and Baseline Metrics

Match ERA5 grid points to SWAN output sites and compute overall validation metrics for the existing (baseline) SWAN run.

In [12]:
# Match ERA5 grid points to SWAN sites
print("Matching ERA5 grid points to SWAN domain...")
matched = match_era5_swan(era5, swan_times, swan_hs, site_locs)
print(f"Matched {len(matched)} ERA5 grid points within SWAN domain")

for m in matched:
    print(f"  ({m['era5_lon']:.1f}\u00b0, {m['era5_lat']:.1f}\u00b0) \u2192 "
          f"Site {m['site_idx']} ({m['n_sites_averaged']} sites averaged), "
          f"{len(m['obs'])} matched timesteps")

# Aggregate all matched pairs and compute overall metrics
all_obs = np.concatenate([m['obs'] for m in matched])
all_sim = np.concatenate([m['sim'] for m in matched])
overall_metrics = validation_metrics(all_sim, all_obs)

print(f"\nOverall Validation Metrics (N = {len(all_obs):,}):")
print(f"{'-'*40}")
for k, v in overall_metrics.items():
    print(f"  {k:>5s} = {v:+.4f}" if k == 'Bias' else f"  {k:>5s} = {v:.4f}")

best_params = {'Cds': 1.36e-5, 'Cf': 0.038, 'gamma': 0.73}

Matching ERA5 grid points to SWAN domain...
Matched 2 ERA5 grid points within SWAN domain
  (-122.5°, 38.0°) → Site 187 (82 sites averaged), 1440 matched timesteps
  (-122.5°, 37.5°) → Site 0 (72 sites averaged), 1440 matched timesteps

Overall Validation Metrics (N = 2,880):
----------------------------------------
   Bias = -0.7592
    MAE = 0.8023
   RMSE = 0.9512
     CC = 0.6158
    NSE = -1.2680
    KGE = 0.4386


### Calibration Results Table

In a full sweep, each row would represent a different SWAN run. Here, the baseline run (marked with a star) is shown alongside perturbed metrics to illustrate the ranking framework. Results are sorted by KGE (highest = best).

In [ ]:
# Build the calibration results table.
#   RUN_SWEEP=True  -> every combination is a real SWAN run (27 runs; slow).
#   RUN_SWEEP=False -> only the baseline row is real; the other 26 rows are
#                      SYNTHETIC perturbations that only illustrate the
#                      ranking layout, and are labelled as such.
import re

BASELINE = (1.36e-5, 0.038, 0.73)   # values in the shipped SWAN inputs
_NUM = r'[-+0-9.eE]+'


def _is_baseline(cds, cf, gm):
    return bool(np.allclose([cds, cf, gm], BASELINE, rtol=1e-6))


def make_calib_input(src, dst, cds, cf, gamma, table_name):
    """
    Copy a SWAN input, set Komen Cds (GEN3 + WCAP), JONSWAP Cf and breaker
    gamma, and redirect the 'sites' TABLE output to table_name. Missing FRIC /
    BREAK commands are inserted above COMPUTE; a missing GEN3 KOMEN raises.
    """
    with open(src) as f:
        txt = f.read()

    def _set(pattern, value, text, required=False, insert_cmd=None):
        new, n = re.subn(pattern, lambda m: m.group(1) + value, text)
        if n:
            return new
        if required:
            raise ValueError(f"Required command not found in {src}: {pattern}")
        if insert_cmd:
            return re.sub(r'(?im)^(\s*COMP)', lambda m: insert_cmd + '\n' + m.group(1),
                          text, count=1)
        return text

    txt = _set(rf'(?im)^(\s*GEN3\s+KOM\w*\s+){_NUM}', f'{cds:.3e}', txt, required=True)
    txt = _set(rf'(?im)^(\s*WCAP\w*\s+KOM\w*\s+){_NUM}', f'{cds:.3e}', txt)
    txt = _set(rf'(?im)^(\s*FRIC\w*\s+JON\w*\s+(?:CON\w*\s+)?){_NUM}', f'{cf:.4f}', txt,
               insert_cmd=f'FRICtion JONswap CONstant {cf:.4f}')
    txt = _set(rf'(?im)^(\s*BRE\w*\s+CON\w*\s+{_NUM}\s+){_NUM}', f'{gamma:.3f}', txt,
               insert_cmd=f'BREaking CONstant 1. {gamma:.3f}')
    txt, n = re.subn(r"(?im)^(\s*TABLE\s+'sites'\s+\w+\s+)'[^']+'",
                     lambda m: m.group(1) + f"'{table_name}'", txt)
    if n != 1:
        raise ValueError(
            f"Expected exactly one TABLE 'sites' line in {src}, found {n}. "
            "Use the ahp_run.swn written by the site-selection notebook.")
    with open(dst, 'w') as f:
        f.write(txt)
    return dst


def metrics_for_table(table_path):
    """Overall SWAN-vs-ERA5 metrics for one TABLE output file."""
    t_times, t_data = load_swan_table(table_path, n_sites)
    mm = match_era5_swan(era5, t_times, t_data['Hsig'], site_locs)
    if not mm:
        raise RuntimeError(f"No ERA5/SWAN matches for {table_path}")
    return validation_metrics(np.concatenate([x['sim'] for x in mm]),
                              np.concatenate([x['obs'] for x in mm]))


all_results = []
if RUN_SWEEP:
    if not swan_available:
        raise RuntimeError("RUN_SWEEP=True but 'swanrun' is not on PATH.")
    if not os.path.exists(SWN_TMPL):
        raise FileNotFoundError(
            f"{SWN_TMPL} not found. Run "
            "notebooks/03_site_selection/swan_coupled_ahp_topsis.ipynb first; "
            "the sweep reuses its POINTS/TABLE set-up so results line up with "
            "candidate_sites.dat.")
    print(f"Running {len(param_combos) - 1} SWAN calibration runs "
          "(baseline reuses site_output.tab) -- this can take many hours.")
    for i, (cds, cf, gm) in enumerate(param_combos, 1):
        if _is_baseline(cds, cf, gm):
            all_results.append({'Cds': cds, 'Cf': cf, 'gamma': gm,
                                **overall_metrics, 'source': 'SWAN run (baseline)'})
            continue
        tag = f'calib_{i:02d}'
        make_calib_input(SWN_TMPL, os.path.join(MODEL_DIR, f'{tag}.swn'),
                         cds, cf, gm, f'{tag}.tab')
        print(f"[{i:2d}/{len(param_combos)}] Cds={cds:.2e} Cf={cf:.3f} gamma={gm:.2f}")
        swan_runner.run_swan(MODEL_DIR, input_name=tag, nproc=4)
        mets = metrics_for_table(os.path.join(MODEL_DIR, f'{tag}.tab'))
        all_results.append({'Cds': cds, 'Cf': cf, 'gamma': gm,
                            **mets, 'source': 'SWAN run'})
else:
    for cds, cf, gm in param_combos:
        if _is_baseline(cds, cf, gm):
            all_results.append({'Cds': cds, 'Cf': cf, 'gamma': gm,
                                **overall_metrics, 'source': 'SWAN run (baseline)'})
            continue
        noise = np.random.RandomState(
            hash((cds, cf, gm)) % 2**31).randn(6) * 0.05
        all_results.append({
            'Cds': cds, 'Cf': cf, 'gamma': gm,
            'Bias':  overall_metrics['Bias']  + noise[0] * 0.3,
            'MAE':   overall_metrics['MAE']   * (1 + abs(noise[1]) * 0.5),
            'RMSE':  overall_metrics['RMSE']  * (1 + abs(noise[2]) * 0.5),
            'CC':    max(0.5, overall_metrics['CC']    - abs(noise[3]) * 0.2),
            'NSE':   max(-1,  overall_metrics['NSE']   - abs(noise[4]) * 0.4),
            'KGE':   max(-1,  overall_metrics['KGE']   - abs(noise[5]) * 0.4),
            'source': 'SYNTHETIC perturbation',
        })

sorted_results = sorted(all_results, key=lambda x: x['KGE'], reverse=True)

print(f"{'-'*100}")
print(f"{'Cds':>10s} {'Cf':>8s} {'g':>5s}  "
      f"{'Bias':>7s} {'MAE':>7s} {'RMSE':>7s} {'CC':>6s} "
      f"{'NSE':>7s} {'KGE':>7s}  source")
print(f"{'-'*100}")
for r in sorted_results[:10]:
    marker = ' *' if r['source'].startswith('SWAN') else ''
    print(f"{r['Cds']:>10.2e} {r['Cf']:>8.3f} {r['gamma']:>5.2f} │ "
          f"{r['Bias']:>+7.3f} {r['MAE']:>7.3f} {r['RMSE']:>7.3f} "
          f"{r['CC']:>6.3f} {r['NSE']:>7.3f} {r['KGE']:>7.3f}  "
          f"[{r['source']}]{marker}")
print(f"{'-'*100}")

real = [r for r in sorted_results if r['source'].startswith('SWAN')]
n_syn = len(sorted_results) - len(real)
print(f"*  = real SWAN run ({len(real)} row(s)).")
if n_syn:
    print(f"    The other {n_syn} rows are SYNTHETIC and ONLY demonstrate the ranking")
    print("    layout. Set RUN_SWEEP=True to replace them with real SWAN runs.")

best = real[0]
print(f"\nBest real run: Cds={best['Cds']:.2e}, Cf={best['Cf']:.3f}, "
      f"gamma={best['gamma']:.2f}  (KGE={best['KGE']:.3f}, {best['source']})")


## 6. Phase 2: Validation Plots

### 6.1 Time Series Comparison

The time series comparison shows how well SWAN reproduces the temporal evolution of wave height at each ERA5 comparison point. Key features to check:
- **Storm peak timing** — does SWAN capture when peaks occur?
- **Storm peak magnitude** — does SWAN over- or under-predict peak $H_s$?
- **Calm periods** — does SWAN match the baseline wave climate?

In [ ]:
matched_sorted = sorted(matched, key=lambda m: len(m['obs']), reverse=True)
n_show = min(4, len(matched_sorted))
sel = matched_sorted[:n_show]

fig, axes = plt.subplots(n_show, 1, figsize=(16, 3.2 * n_show),
                         sharex=True, squeeze=False)
axes = axes.flatten()
fig.suptitle(f'Model Validation --  SWAN vs ERA5 Time Series  ({cfg.display_name})',
             fontsize=14, fontweight='bold')

for i, (m, ax) in enumerate(zip(sel, axes)):
    mets = validation_metrics(m['sim'], m['obs'])

    ax.plot(m['times'], m['obs'], 'b-', lw=0.8, alpha=0.8,
            label='ERA5 (reference)')
    ax.plot(m['times'], m['sim'], 'r-', lw=0.8, alpha=0.8,
            label='SWAN (simulated)')
    ax.set_ylabel('$H_s$ (m)', fontsize=10)
    ax.legend(fontsize=8, loc='upper right', ncol=2)
    ax.grid(True, alpha=0.2, ls='--')
    ax.set_ylim(bottom=0)
    ax.set_title(
        f'ERA5 Point ({m["era5_lon"]:.1f}°, {m["era5_lat"]:.1f}°)  |  '
        f'CC={mets["CC"]:.3f}, RMSE={mets["RMSE"]:.3f} m, '
        f'KGE={mets["KGE"]:.3f}',
        fontsize=10, fontweight='bold')

axes[-1].set_xlabel('Date', fontsize=10)
span_days = (cfg.t_end - cfg.t_start).days
if span_days > 14:
    axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
else:
    axes[-1].xaxis.set_major_locator(mdates.DayLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
fig.autofmt_xdate(rotation=30)

plt.tight_layout(rect=[0, 0, 1, 0.97])
_fname = f"validation_{cfg.key}_cell20.png"
plt.savefig(os.path.join(FIGS_DIR, _fname), dpi=140, bbox_inches='tight')
plt.show()


### 6.2 Scatter Plot

The scatter plot provides a comprehensive view of model performance across all matched data points. Points clustering tightly along the 1:1 line indicate good agreement. The 2-D histogram (log-scale color) reveals the density of observations, helping to distinguish systematic biases from random scatter.

Statistical metrics are annotated directly on the plot for reference.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# 2D histogram for density
h = ax.hist2d(all_obs, all_sim, bins=60, cmap='YlOrRd',
              cmin=1, norm=plt.matplotlib.colors.LogNorm())
plt.colorbar(h[3], ax=ax, label='Count', shrink=0.85)

lim = [0, max(np.percentile(all_obs, 99.5),
              np.percentile(all_sim, 99.5)) * 1.1]
ax.plot(lim, lim, 'k--', lw=1.2, label='1:1 line')
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.set_xlabel('ERA5 $H_s$ (m) — reference', fontsize=11)
ax.set_ylabel('SWAN $H_s$ (m) — simulated', fontsize=11)

txt = (
    f"N       = {len(all_obs):,}\n"
    f"Bias    = {overall_metrics['Bias']:+.3f} m\n"
    f"MAE     = {overall_metrics['MAE']:.3f} m\n"
    f"RMSE    = {overall_metrics['RMSE']:.3f} m\n"
    f"CC      = {overall_metrics['CC']:.3f}\n"
    f"NSE     = {overall_metrics['NSE']:.3f}\n"
    f"KGE     = {overall_metrics['KGE']:.3f}"
)
ax.text(0.03, 0.97, txt, transform=ax.transAxes, va='top',
        fontsize=10, family='monospace',
        bbox=dict(facecolor='white', alpha=0.85, ec='none'))

ax.grid(True, alpha=0.15, ls='--')
ax.set_title(f'SWAN vs ERA5 --  {cfg.display_name} (all matched points)',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
_fname = f"validation_{cfg.key}_cell22.png"
plt.savefig(os.path.join(FIGS_DIR, _fname), dpi=140, bbox_inches='tight')
plt.show()


### 6.3 Taylor Diagram

The Taylor diagram (Taylor, 2001) summarises three statistics simultaneously in a single polar plot:
- **Radial distance** = normalised standard deviation ($\sigma_{sim} / \sigma_{obs}$)
- **Angular position** = correlation coefficient ($r = \cos\theta$)
- **Distance from reference point** = centred RMS difference

The **reference point** (black dot at $r=1$, $\theta=0$) represents a perfect model. Each marker represents a different ERA5 comparison point. Points closer to the reference indicate better agreement.

Gray concentric arcs show contours of constant normalised RMSD.

In [ ]:
fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(111, projection='polar')

ax.set_thetamin(0)
ax.set_thetamax(90)
ax.set_theta_direction(-1)
ax.set_theta_zero_location('N')

# Reference point (perfect model)
ax.plot(0, 1, 'ko', ms=10, label='Reference (perfect)', zorder=5)

# RMSD contours centered on reference point
rmsd_levels = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5]
for rms_val in rmsd_levels:
    theta_c = np.linspace(0, np.pi / 2, 180)
    rad_c = np.sqrt(1 + rms_val ** 2 - 2 * rms_val * np.cos(theta_c - np.pi / 2))
    ax.plot(theta_c, rad_c, ls=':', color='grey', lw=0.8, alpha=0.7)
    # Label the contour near the top arc
    ax.text(np.pi / 2 * 0.2, np.sqrt(1 + rms_val**2),
            f'{rms_val:.2f}', fontsize=8, color='grey', rotation=-40)

# Data points (one per matched ERA5 grid cell)
cmap = plt.colormaps.get_cmap('viridis')
for i, m in enumerate(matched):
    sig_o = np.std(m['obs'])
    sig_s = np.std(m['sim'])
    cc    = np.corrcoef(m['obs'], m['sim'])[0, 1]
    theta = np.arccos(np.clip(cc, -1, 1))
    r     = sig_s / sig_o if sig_o > 0 else 0.0
    ax.plot(theta, r, 'o', ms=9, color=cmap(i / max(1, len(matched) - 1)),
            mec='k', mew=0.6,
            label=f'({m["era5_lon"]:.1f}°, {m["era5_lat"]:.1f}°)')

ax.set_rlim(0, 1.5)
ax.set_rticks([0.25, 0.5, 0.75, 1.0, 1.25])
ax.set_rlabel_position(135)

# Correlation ticks on the angular axis
cc_ticks = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.99])
theta_ticks = np.arccos(cc_ticks)
ax.set_xticks(theta_ticks)
ax.set_xticklabels([f'{v:.2f}' for v in cc_ticks])

ax.set_title(f'Taylor Diagram --  SWAN vs ERA5 ({cfg.display_name})',
             fontsize=13, fontweight='bold', pad=22)
ax.legend(loc='lower left', bbox_to_anchor=(1.05, 0.0), fontsize=8,
          framealpha=0.9)
plt.tight_layout()
_fname = f"validation_{cfg.key}_cell24.png"
plt.savefig(os.path.join(FIGS_DIR, _fname), dpi=140, bbox_inches='tight')
plt.show()


### 6.4 Metrics Summary + Diagnostics

Bar charts showing each validation metric broken down by ERA5 comparison point. The dashed black line shows the overall (aggregated) metric, and green dotted lines show commonly used thresholds for acceptable performance.

Following the bar charts, an automated diagnostics block inspects the overall metrics and prints specific calibration recommendations (e.g. "positive bias: try higher ``WHITEcapping`` dissipation"). This lets the notebook translate numbers into actionable guidance.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Model Validation --  Statistical Metrics Summary ({cfg.display_name})',
             fontsize=14, fontweight='bold')

metric_names = ['Bias', 'MAE', 'RMSE', 'CC', 'NSE', 'KGE']
bar_colors = ['#d6604d', '#f4a582', '#b2182b', '#4393c3', '#2166ac', '#053061']
thresholds = {'CC': 0.90, 'NSE': 0.70, 'KGE': 0.50}

per_point = [validation_metrics(m['sim'], m['obs']) for m in matched]

for ax, mname, color in zip(axes.flat, metric_names, bar_colors):
    vals = [pp[mname] for pp in per_point]
    labels = [f"({m['era5_lon']:.1f}, {m['era5_lat']:.1f})" for m in matched]
    bars = ax.bar(range(len(vals)), vals, color=color, ec='k', lw=0.6)
    ax.axhline(overall_metrics[mname], ls='--', color='black', lw=1,
               label=f"Overall ({overall_metrics[mname]:+.3f})")
    if mname in thresholds:
        ax.axhline(thresholds[mname], ls=':', color='green', lw=1,
                   label=f'Threshold ({thresholds[mname]})')
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel(mname, fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.2, ls='--', axis='y')
    ax.legend(fontsize=8, loc='best')
    ax.set_title(f'{mname} per ERA5 point', fontsize=10)

plt.tight_layout(rect=[0, 0, 1, 0.95])
_fname = f"validation_{cfg.key}_cell26.png"
plt.savefig(os.path.join(FIGS_DIR, _fname), dpi=140, bbox_inches='tight')
plt.show()

# ---------- Diagnostics: translate metrics into calibration advice ----------
print()
print("=" * 70)
print(f"  Automated diagnostics for {cfg.display_name}")
print("=" * 70)

b    = overall_metrics['Bias']
rmse = overall_metrics['RMSE']
cc   = overall_metrics['CC']
nse  = overall_metrics['NSE']
kge  = overall_metrics['KGE']

if abs(b) < 0.1:
    print(f"  [OK]     Mean bias {b:+.3f} m is small (|Bias| < 0.10 m).")
elif b > 0:
    print(f"  [WARN]   Positive bias {b:+.3f} m -- SWAN over-predicts Hs.")
    print("           Try: raise Cds (whitecapping) or check wind forcing magnitude.")
else:
    print(f"  [WARN]   Negative bias {b:+.3f} m -- SWAN under-predicts Hs.")
    print("           Try: lower Cds, increase wind drag, verify boundary TPARs.")

if cc >= 0.90:
    print(f"  [OK]     CC  = {cc:.3f}  (>= 0.90)")
elif cc >= 0.75:
    print(f"  [MED]    CC  = {cc:.3f}  (0.75-0.90) -- peak timing acceptable but can improve.")
else:
    print(f"  [POOR]   CC  = {cc:.3f}  (< 0.75) -- check forcing temporal resolution "
          "and boundary update frequency.")

if nse >= 0.70:
    print(f"  [OK]     NSE = {nse:.3f}  (>= 0.70)")
elif nse >= 0.40:
    print(f"  [MED]    NSE = {nse:.3f}  (0.40-0.70) -- variance partially captured.")
else:
    print(f"  [POOR]   NSE = {nse:.3f}  (< 0.40) -- significant residual variance; "
          "consider finer grid or more accurate bathymetry.")

if kge >= 0.75:
    print(f"  [OK]     KGE = {kge:.3f}  (>= 0.75)")
elif kge >= 0.50:
    print(f"  [MED]    KGE = {kge:.3f}  (0.50-0.75) -- usable but room to improve.")
else:
    print(f"  [POOR]   KGE = {kge:.3f}  (< 0.50) -- run a full Cds/Cf/gamma sweep.")

print("=" * 70)
# --- Persist validation outputs to figs/06_model_validation/<region>/ ---
import csv
metrics_csv = os.path.join(FIGS_DIR, f'validation_metrics_{cfg.key}.csv')
with open(metrics_csv, 'w', newline='') as _f:
    w = csv.writer(_f)
    w.writerow(['era5_lon', 'era5_lat', 'n_pairs',
                'Bias', 'MAE', 'RMSE', 'CC', 'NSE', 'KGE'])
    for m, pp in zip(matched, per_point):
        w.writerow([m['era5_lon'], m['era5_lat'], len(m['obs']),
                    pp['Bias'], pp['MAE'], pp['RMSE'],
                    pp['CC'],   pp['NSE'], pp['KGE']])
    w.writerow([])
    w.writerow(['OVERALL', '', len(all_obs),
                overall_metrics['Bias'], overall_metrics['MAE'],
                overall_metrics['RMSE'], overall_metrics['CC'],
                overall_metrics['NSE'], overall_metrics['KGE']])
print(f'Wrote per-point metrics -> {metrics_csv}')


## Summary

This notebook demonstrated a complete model calibration and validation workflow for the SWAN wave model:

| Phase | Component | Purpose |
|-------|-----------|--------|
| **Calibration** | Parameter sweep framework | Systematically vary $C_{ds}$, $C_f$, $\gamma$ to minimise error |
| **Calibration** | Results ranking table | Identify optimal parameter set by KGE |
| **Validation** | Time series comparison | Visual check of peak timing and magnitude |
| **Validation** | Scatter plot | Overall bias and spread assessment |
| **Validation** | Taylor diagram | Simultaneous view of CC, std ratio, and RMSD |
| **Validation** | Metrics summary | Quantitative per-point and overall performance |

### Key Functions Used

- `swan_runner.read_table()` — parses SWAN TABLE text output
- `validation_metrics()` — computes Bias, MAE, RMSE, CC, NSE, KGE
- `match_era5_swan()` — spatially matches ERA5 grid points to SWAN output sites

### Interpretation Guide

- **CC > 0.90** indicates the model captures the timing of wave events well
- **Bias close to 0** means no systematic over- or under-prediction
- **KGE > 0.50** is generally considered acceptable for wave models
- **Taylor diagram points near the reference** represent near-perfect model skill

### References

- Taylor, K.E. (2001). Summarizing multiple aspects of model performance in a single diagram. *J. Geophys. Res.*, 106(D7), 7183–7192.
- Gupta, H.V., et al. (2009). Decomposition of the mean squared error and NSE performance criteria. *J. Hydrology*, 377(1-2), 80–91.